# 7. Команды и успех: опыт и посылки
Добавляем к размеру команды опыт участников и число посылок (признаки из `src/features.py`).
Цель на будущее (модель): попадание в топ-10% приватного лидерборда.

In [ ]:
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from src.data_loader import load
from src.features import team_features, submission_counts

sns.set_theme(style="whitegrid")
comp = load("Competitions")
teams = load("Teams")
members = load("TeamMemberships")
users = load("Users")

In [ ]:
featured = comp.loc[comp["HostSegmentTitle"] == "Featured", ["Id", "TotalTeams"]]
df = teams.merge(featured, left_on="CompetitionId", right_on="Id", suffixes=("", "_comp"))
df = df.dropna(subset=["PrivateLeaderboardRank"]).set_index("Id")
df["rank_pct"] = df["PrivateLeaderboardRank"] / df["TotalTeams"]
df["top10"] = df["rank_pct"] <= 0.1

Submissions читается по частям — ячейка может работать долго.

In [ ]:
df = df.join(team_features(teams, members, users)).join(submission_counts(teams, comp))
df[["team_size", "max_prev_comps", "max_tier", "n_subs", "n_subs_first7d"]].describe()

## Корреляции с местом (Спирмен)

In [ ]:
cols = ["team_size", "max_prev_comps", "mean_prev_comps", "max_tier", "n_subs", "n_subs_first7d"]
df[cols + ["rank_pct"]].corr(method="spearman")["rank_pct"].drop("rank_pct")

## Доля топ-10% по группам

In [ ]:
df["exp_group"] = pd.cut(df["max_prev_comps"], [-1, 0, 2, 5, 10, float("inf")],
                         labels=["0", "1-2", "3-5", "6-10", "10+"])
df.groupby("exp_group", observed=True)["top10"].mean().plot(
    kind="bar", figsize=(8, 4), title="Доля попаданий в топ-10% по опыту (прошлых соревнований у самого опытного)"
)
plt.savefig("../reports/figures/07_top10_by_experience.png", bbox_inches="tight")
plt.show()

In [ ]:
df["subs_group"] = pd.qcut(df["n_subs_first7d"].fillna(0), 5, duplicates="drop")
df.groupby("subs_group", observed=True)["top10"].mean()

TODO: эти признаки + размер команды — основа датасета для модели (блок 5).